# Расчет коэффициентов пролонгации. Отдел сопровождения клиентов

# 1. Загрузка данных

In [1]:
from pathlib import Path

from src.data_preparation import load_data, prepare_data
from src.prolongation_calculator import calculate_prolongation_report, coef
from src.excel_report import build_excel_report

root = Path(".").resolve()
prolongations, financial_data = load_data(root)

print("prolongations:", prolongations.shape)
print("financial_data:", financial_data.shape)

prolongations: (477, 3)
financial_data: (451, 19)


# 2. Подготовка данных
Нормализуем месяцы, деньги и менеджеров. Подробная логика вынесена в src/data_preparation.py.

In [2]:

prolongations, financial_data, month_cols, period_to_col, col_to_period = prepare_data(
    prolongations,
    financial_data,
)

print("Месячных колонок:", len(month_cols))
print("Период данных:", min(period_to_col), "—", max(period_to_col))

Месячных колонок: 16
Период данных: 2022-11 — 2024-02


# 3. Правила обработки специальных значений

В расчёте используются правила из ТЗ:  
    - `стоп` / `end` в последний месяц реализации или раньше исключают проект из пролонгаций;  
    - пустой месяц означает отсутствие отгрузки;  
    - `в ноль` и месяцы, где все части оплаты равны 0, подтягивают предыдущую эффективную отгрузку.  

# 4. Источник менеджера

Для группировок и отчёта используется `AM` из `prolongations.csv`,  
потому что по ТЗ этот источник первичен относительно `Account` в `financial_data.csv`.

# 5. Расчёт коэффициентов пролонгации
Подробная логика агрегации финансов, исключений и K1/K2 вынесена в src/prolongation_calculator.py.

In [3]:
result = calculate_prolongation_report(
    prolongations=prolongations,
    financial_data=financial_data,
    month_cols=month_cols,
    col_to_period=col_to_period,
    year=2023,
)

print(result.fin_long.head())
print("строк fin_long:", len(result.fin_long))
print("Исключено по стоп/end (id):", len(result.exclude_stop_set))

   id   period    sum_num  has_stop  has_end  all_parts_zero_like
0  15  2022-11  439280.00     False    False                False
1  15  2022-12  439280.00     False    False                False
2  15  2023-01  102433.75     False    False                False
3  15  2023-02  102433.75     False    False                False
4  15  2023-03  102433.75     False    False                False
строк fin_long: 5024
Исключено по стоп/end (id): 36


# 6. Формирование Excel-отчёта

In [6]:
report_path = build_excel_report(result, root / "reports" / "report_prolongation_2023.xlsx")

print("Файл:", report_path.relative_to(root).as_posix())
print(
    "Отдел за 2023 (год): K1 =",
    coef(result.year_dept_k1["num"], result.year_dept_k1["den"]),
    " K2 =",
    coef(result.year_dept_k2["num"], result.year_dept_k2["den"]),
)

Файл: reports/report_prolongation_2023.xlsx
Отдел за 2023 (год): K1 = 0.4743  K2 = 0.0642
